In [5]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [6]:
device = 0 if torch.backends.mps.is_available() else -1

In [7]:
classifier = pipeline(
    "text-classification", 
    model="bhadresh-savani/distilbert-base-uncased-emotion", 
    top_k=None,
    device=device 
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 20139.79it/s]


In [8]:
def get_emotion_vector(text):
    clean_text = str(text)[:512]
    results = classifier(clean_text)
    return [res['score'] for res in results[0]]

In [9]:
df = pd.read_csv("../../../data/cleaned_data.csv")
features = []
for text in tqdm(df['cleaned_statement']):
    features.append(get_emotion_vector(text))

100%|██████████| 31993/31993 [07:01<00:00, 75.97it/s] 


In [10]:
emotion_cols = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
features_df = pd.DataFrame(features, columns=emotion_cols)
features_df['label'] = df['label']

In [11]:
features_df.to_csv('../../../data/features_for_random_forest.csv', index=False)